# 02. 高级量化方法与配置

本章梳理 Quark for PyTorch LLM 量化实战前需要理解的配置概念。本教程聚焦 PyTorch LLM 流程，不展开 ONNX 量化流程。

本章目标：

- 理解 `LLMTemplate` 如何生成 LLM 量化配置。
- 理解 SmoothQuant、AWQ、GPTQ、Rotation 等高级 PTQ 算法的适用场景。
- 理解从底层对象配置 PyTorch 量化时，需要关注哪些维度。


## 1. LLMTemplate：LLM 量化配置的快捷入口

对于 LLM，Quark 推荐通过 `LLMTemplate` 创建量化配置。它封装了常见模型结构的层名、KV cache 层、默认排除层和算法配置。

典型流程：

```python
from quark.torch import LLMTemplate

template = LLMTemplate.get("llama")
config = template.get_config(
    scheme="mxfp4",
    algorithm="smoothquant",
)
```

相比从 `QTensorConfig`、`QLayerConfig`、`QConfig` 逐层手写，`LLMTemplate` 更适合 LLM 的常见 PTQ 流程。


## 2. LLMTemplate 支持的常见量化方案

Quark 的 LLM 配置文档列出了一组常用 scheme。课程中需要重点理解以下几类：

| scheme | 描述 | 常见用途 |
| --- | --- | --- |
| `int4_wo_128` / `int4_wo_64` / `int4_wo_32` | INT4 weight-only，对称，per-group | AWQ / 低显存推理 |
| `uint4_wo_128` / `uint4_wo_64` / `uint4_wo_32` | UINT4 weight-only，非对称，per-group | AWQ / 权重量化 |
| `int8` | INT8，per-tensor，静态量化 | W8A8 / SmoothQuant 对照 |
| `fp8` | FP8 E4M3，per-tensor，静态量化 | 支持 FP8 kernel 的 GPU 推理 |
| `ptpc_fp8` | weight per-channel static，activation per-token dynamic | 面向 vLLM 优化的 FP8 路线 |
| `mxfp4` | OCP MXFP4，per-group / block scaling，group size 32 | MXFP4 权重量化与 vLLM 验证 |

本教程第三章主线使用：

- `mxfp4 + smoothquant`
- `model_export=hf_format`
- vLLM `--quantization quark` 验证


## 3. 高级量化算法怎么选？

这些算法不是新的 `ModelQuantizer`，而是通过 `QConfig` / `LLMTemplate.get_config(..., algorithm=...)` 挂接到量化流程中的优化步骤。

| 方法 | 解决的问题 | 适合场景 |
| --- | --- | --- |
| **SmoothQuant** | 平滑 activation outlier，并把部分量化难度迁移到 weight | activation 参与量化或低 bit 流程对 outlier 敏感时 |
| **AutoSmoothQuant** | 自动搜索更合适的 smoothing scale | 不想手动选择 SmoothQuant alpha 时 |
| **AWQ** | 低 bit 权重量化时保护重要通道 | INT4 / UINT4 weight-only 推理 |
| **GPTQ** | 用二阶信息做列级误差补偿 | 更重但通常更准的 weight-only PTQ |
| **Qronos / GPTAQ** | Hessian 族高级变体 | 高精度恢复或研究场景 |
| **Rotation / QuaRot** | 通过旋转降低 outlier | 极低 bit、KV cache、W8A8 outlier 场景 |

第三章聚焦 SmoothQuant 与 MXFP4 的组合，并通过 `quantize_quark.py` 完成导出、评测和 vLLM 验证。


## 4. 一个高级 LLMTemplate 配置长什么样？

下面例子展示 `LLMTemplate.get_config` 的常见能力：全局 scheme、高级算法、KV cache、attention、按层覆盖和排除层。

```python
from torch import nn
from quark.torch import LLMTemplate

llama_template = LLMTemplate.get("llama")

config = llama_template.get_config(
    scheme="mxfp4",                # 全局量化方案
    algorithm="smoothquant",       # 高级量化算法
    kv_cache_scheme="fp8",         # 可选：KV cache 量化
    min_kv_scale=1.0,              # 可选：KV cache scale 下限
    attention_scheme="fp8",        # 可选：attention 量化
    layer_config={                 # 可选：按层名覆盖量化方案
        "*.mlp.gate_proj": "mxfp4",
        "*.mlp.up_proj": "mxfp4",
        "*.mlp.down_proj": "mxfp4",
    },
    layer_type_config={            # 可选：按层类型覆盖量化方案
        nn.LayerNorm: "fp8",
    },
    exclude_layers=["lm_head"],    # 可选：排除不量化的层
)
```

需要注意：

- layer-wise / layer-type-wise 配置会覆盖 global scheme。
- 如果传入多个高级算法，Quark 会按列表顺序执行。
- 某个 scheme 是否适合实际部署，还取决于导出格式和推理框架是否支持。


## 5. 从零配置 PyTorch 量化：四层对象

如果不用 `LLMTemplate`，Quark 的 PyTorch 量化配置可以从底层对象开始搭：

1. **`QTensorConfig`**：描述某个 tensor 怎么量化，例如 `dtype`、observer、qscheme、是否 dynamic、是否 symmetric、group size。
2. **`QLayerConfig`**：描述某类 module 的输入、输出、weight、bias 分别使用什么 tensor quant config。
3. **`AlgoConfig`**：可选，高级算法配置，例如 `AWQConfig`、`SmoothQuantConfig`、`GPTQConfig`、`RotationConfig`、`QronosConfig`。
4. **`QConfig`**：模型级总配置，组合 global config、layer config、exclude、algo config 等。

直觉上：

```text
QTensorConfig  ->  一个 tensor 怎么量化
QLayerConfig   ->  一个 layer 的哪些 tensor 要量化
AlgoConfig     ->  量化前/量化中要不要做高级优化
QConfig        ->  整个模型的量化方案
```

## 6. Calibration Methods：校准方法

校准的目标是用代表性数据估计量化参数，例如 min/max、scale、zero point 或 block scale。

常见方法：

- **MinMax**：记录 tensor 的运行最小值和最大值，用于计算量化参数；适合作为基线。
- **Percentile**：基于 histogram 的分位数估计缩放范围，减少极端 outlier 对 scale 的影响。
- **MSE**：搜索使量化前后误差更小的量化参数，更关注重构误差。

在 LLM 中，activation outlier 会显著影响低 bit 量化。SmoothQuant、AWQ 等算法的价值之一，就是降低这些 outlier 对量化参数的影响。


## 7. Calibration Datasets：校准数据集

Quark 使用 PyTorch `DataLoader` 进行校准。校准数据通常不需要标签，但应尽量代表真实推理输入分布。

常见输入形式包括：

- `torch.Tensor`
- `list[torch.Tensor]`
- `list[dict[str, torch.Tensor]]`
- `dict[str, torch.Tensor]`

LLM 示例中常见校准集包括：

- `pileval`
- `wikitext`
- `pileval_for_awq_benchmark`
- `wikitext_for_gptq_benchmark`

教程命令会用较小的 `num_calib_data` 验证流程；正式实验应根据精度和时间预算增加校准样本数和序列长度。


## 8. Quantization Strategies：量化策略

Quark PyTorch 文档把 PTQ 策略分成三类：

- **Post Training Weight-Only Quantization**：提前量化权重，推理时 activation 保持浮点类型。AWQ / INT4 weight-only 属于这类。
- **Post Training Static Quantization**：权重和 activation 都量化，需要校准数据确定 activation 量化参数。
- **Post Training Dynamic Quantization**：权重提前量化，activation 在运行时动态量化，适合 activation 分布难以提前固定的场景。

本教程第三章使用 `mxfp4 + SmoothQuant`。其中 MXFP4 负责低 bit 权重表示，SmoothQuant 用于平滑 activation outlier，降低量化误差。


## 9. Quantization Schemes：量化粒度

常见量化粒度包括：

- **per-tensor**：整个 tensor 共用一个 scale。配置简单，但对 outlier 敏感。
- **per-channel**：每个 channel 单独一组量化参数。通常更准确，但会增加量化参数。
- **per-group / per-block**：把 tensor 切成小 group 或 block 分别量化，在压缩率和精度之间折中。

例如，`int4_wo_128` 中的 `128` 表示 group size；MXFP4 则使用 block scaling，每个 block 维护共享 scale。


## 10. Quantization Symmetry：对称与非对称

对整数类型来说，量化还要考虑对称性：

- **symmetric quantization**：zero point 通常是 0，公式更简单，硬件实现友好。
- **asymmetric quantization**：zero point 可以偏移，让量化后的 0 表示真实值中不一定为 0 的位置，能更灵活覆盖非对称分布。

课程里会见到：

- `int4_wo_*`：对称 INT4 weight-only。
- `uint4_wo_*`：非对称 UINT4 weight-only。

## 11. 本章小结

进入实战前，先记住这条配置路径：

```text
目标硬件 / 推理框架
  -> 选择 quantization strategy
  -> 选择 scheme / dtype / symmetry / granularity
  -> 选择 calibration method 和 calibration dataset
  -> 可选高级算法：SmoothQuant / AWQ / GPTQ / Rotation
  -> 用 LLMTemplate 或底层 QTensorConfig / QLayerConfig / QConfig 构建配置
  -> ModelQuantizer.quantize_model()
  -> 导出为目标推理框架可读取的格式
```

下一章进入 SmoothQuant MXFP4 实战：使用 `quantize_quark.py` 量化 `Qwen2.5-7B-Instruct`，对比模型大小和评测结果，并用 vLLM 验证导出模型。
